# Test Attendance Records with Proof Images

This notebook tests creating attendance records with proof images (camera frames).

**Features tested:**
- Creating attendance records with proof images
- Source labels: automatic, manual, from_unrecognized
- Uploading images as multipart/form-data
- Verifying proof_image_url is returned

**Before running:**
1. Make sure backend is running
2. Have some test images ready (jpg/png)
3. Login as ORG ADMIN

## Setup

In [ ]:
import requests
import os
from datetime import datetime, timezone
from dotenv import load_dotenv
from PIL import Image, ImageDraw, ImageFont
import io

load_dotenv()

BACKEND_URL = os.getenv('SO_BACKEND_API_URL', 'http://localhost:7091')
session = requests.Session()
session.headers.update({"accept": "application/json"})

print('Setup complete!')
print(f'Backend API: {BACKEND_URL}')

Setup complete!
Backend API: http://localhost:7091


## Login

In [ ]:
def login_to_backend(email=None, password=None, client_slug='humblebee'):
    if email is None:
        email = os.getenv('SO_ADMIN_EMAIL', 'admin@humblebee.ai')
    if password is None:
        password = os.getenv('SO_ADMIN_PASSWORD', 'admin123')

    print(f'Logging in as {email} to org "{client_slug}"...')

    response = session.post(
        f'{BACKEND_URL}/api/auth/login',
        json={'email': email, 'password': password, 'client_slug': client_slug}
    )

    if response.status_code == 200:
        data = response.json()
        token = data.get('token') or data.get('accessToken')
        session.headers.update({'Authorization': f'Bearer {token}'})
        print('Login successful')
        return token, client_slug
    else:
        print(f'Login failed: {response.status_code}')
        return None, None

auth_token, slug = login_to_backend(
    email='humblebee@gmail.com',
    password='Humblebee2025@',
    client_slug='humblebee'
)

Logging in as humblebee@gmail.com to org "humblebee"...
Login successful


## Get Users

In [ ]:
def list_org_users(slug, page=1, limit=50):
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/users", params={"page": page, "limit": limit})
    r.raise_for_status()
    return r.json()

users = list_org_users(slug)
print(f"Found {len(users)} users")
for u in users[:5]:
    print(f"  ID: {u.get('id')}, Name: {u.get('full_name')}")

Found 40 users
  ID: 80, Name: Integration Test User 08:47:23
  ID: 79, Name: Integration Test User 08:45:30
  ID: 78, Name: Integration Test User 08:30:15
  ID: 77, Name: Integration Test User 08:28:52
  ID: 71, Name: Unrecognized guy


## Create Test Image

In [ ]:
def create_test_image(text="Test Proof", size=(640, 480)):
    img = Image.new('RGB', size, color=(30, 60, 120))
    draw = ImageDraw.Draw(img)

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    text_with_time = f"{text}\n{timestamp}"

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 30)
    except:
        font = ImageFont.load_default()

    draw.text((50, 200), text_with_time, fill=(255, 255, 255), font=font)

    img_bytes = io.BytesIO()
    img.save(img_bytes, format='JPEG')
    img_bytes.seek(0)
    return img_bytes

test_image = create_test_image()
print(f'Test image created: {len(test_image.getvalue())} bytes')

Test image created: 10825 bytes


In [ ]:
# Helper function for ISO timestamp
def iso_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

print('Helper functions ready!')

Helper functions ready!


## Test 1: Record with Proof (Automatic)

In [19]:
user_id = 61
status = "in"

proof_image = create_test_image(f"Auto Entry - User {user_id}")

files = {'proof_image': ('proof.jpg', proof_image, 'image/jpeg')}
data = {
    'user_id': str(user_id),
    'status': status,
    'timestamp': iso_now(),
    'source': 'automatic'
}

print(f'Creating AUTOMATIC record with proof...')
headers = {k: v for k, v in session.headers.items() if k.lower() != 'content-type'}
r = requests.post(f"{BACKEND_URL}/api/org/{slug}/attendance-records", headers=headers, files=files, data=data)

if r.status_code == 201:
    result = r.json()
    print('Record created successfully!')
    print(f'  ID: {result.get("id")}')
    print(f'  Source: {result.get("source")}')
    print(f'  Proof URL: {result.get("proof_image_url")}')
else:
    print(f'Error: {r.status_code}')
    print(r.text[:300])

Creating AUTOMATIC record with proof...
Record created successfully!
  ID: 13492
  Source: automatic
  Proof URL: https://storage.googleapis.com/hbai-general-data/2025/cv.face-recognition/humblebee/attendance_proofs/proof_61_1763896816554_eb572d6e-d3f3-4d33-b4de-78de373bf9e0.jpg?GoogleAccessId=model-uploader%40humblebee-project.iam.gserviceaccount.com&Expires=1764501618&Signature=TOBvsumUK4YCjnDrnOtpfxizUBSzd%2B%2Fnc5VHN5rOoX2x%2BETmWt9CLru0vq1%2FSJvZwoXyvsUB6zQ10zm2aU8qDE1FgF8f2VgYmItU8omwH1OgOs3NALSv01ZmbtgbucWd9vvDGUUiv3iHWjJxDQr61bJN%2FyDKdT7Howw1kEPFWzA73iUPPe4rMy5SsktRf75Ag4jIblkPP3bjy0c%2FIj6uzGjEOWMzT0XTR2UxjqWujvAQPvLyRzlqGEM04e%2BFAh2Lei%2BrdZcwgMgyLGp1xfvjBjmyuQMumiXFquPY07ju8TiwyFhpIMnMglNJjdlJGDiJ6239%2F%2BgzV859mVH61qL0mQ%3D%3D


## Test 2: Manual Record (No Proof)

In [20]:
user_id = 61
status = "out"

payload = {
    'user_id': int(user_id),
    'status': status,
    'timestamp': iso_now(),
    'source': 'manual'
}

print('Creating MANUAL record (no proof)...')
session.headers.update({'Content-Type': 'application/json'})
r = session.post(f"{BACKEND_URL}/api/org/{slug}/attendance-records", json=payload)

if r.status_code == 201:
    result = r.json()
    print('Manual record created!')
    print(f'  ID: {result.get("id")}')
    print(f'  Source: {result.get("source")}')
    print(f'  Proof: {result.get("proof_image_url") or "None"}')
else:
    print(f'Error: {r.status_code}')
    print(r.text[:300])

Creating MANUAL record (no proof)...
Manual record created!
  ID: 13493
  Source: manual
  Proof: None


## Test 3: Multiple Records

In [21]:
import time

scenarios = [
    {'source': 'automatic', 'with_image': True, 'status': 'in'},
    {'source': 'manual', 'with_image': False, 'status': 'out'},
    {'source': 'from_unrecognized', 'with_image': True, 'status': 'in'}
]

print('Creating multiple test records...')

for i, scenario in enumerate(scenarios, 1):
    user = users[i % len(users)]
    print(f'{i}. {scenario["source"]} - {scenario["status"]} - {user["full_name"]}')

    if scenario['with_image']:
        proof_image = create_test_image(f"{scenario['source']} - {user['full_name']}")
        files = {'proof_image': ('proof.jpg', proof_image, 'image/jpeg')}
        data = {
            'user_id': str(user['id']),
            'status': scenario['status'],
            'timestamp': iso_now(),
            'source': scenario['source']
        }
        headers = {k: v for k, v in session.headers.items() if k.lower() != 'content-type'}
        r = requests.post(f"{BACKEND_URL}/api/org/{slug}/attendance-records", headers=headers, files=files, data=data)
    else:
        payload = {
            'user_id': int(user['id']),
            'status': scenario['status'],
            'timestamp': iso_now(),
            'source': scenario['source']
        }
        session.headers.update({'Content-Type': 'application/json'})
        r = session.post(f"{BACKEND_URL}/api/org/{slug}/attendance-records", json=payload)

    if r.status_code == 201:
        result = r.json()
        print(f'  Created - ID: {result.get("id")}')
    else:
        print(f'  Failed: {r.status_code}')

    time.sleep(0.5)

print('\nAll test records created!')
print('Check the History page to see proof images and source labels')

Creating multiple test records...
1. automatic - in - Integration Test User 08:45:30
  Created - ID: 13494
2. manual - out - Integration Test User 08:30:15
  Created - ID: 13495
3. from_unrecognized - in - Integration Test User 08:28:52
  Created - ID: 13496

All test records created!
Check the History page to see proof images and source labels


## Verify Records

In [ ]:
today = datetime.now().strftime('%Y-%m-%d')
print(f'Fetching records for {today}...')

r = session.get(f"{BACKEND_URL}/api/org/{slug}/attendance-records", params={'date': today, 'page': 1, 'limit': 20})

if r.status_code == 200:
    data = r.json()
    records = data.get('records', [])
    print(f'\nFound {len(records)} records\n')

    for rec in records[:10]:
        user_name = next((u['full_name'] for u in users if u['id'] == rec['user_id']), 'Unknown')
        proof = 'Yes' if rec.get('proof_image_url') else 'No'
        source = rec.get('source', 'automatic')
        print(f"{rec['timestamp'][11:19]} | {rec['status']:4} | {source:20} | Proof: {proof} | {user_name}")
else:
    print(f'Failed: {r.status_code}')